# Assignment — Parallel Processing of Iranian Web Logs

You are given 20 Apache web-log files and one large file containing the same data.

Your task is to parse the 20 part files concurrently, combine the results, sort all valid events by timestamp, save them as CSV, and compare the runtime with parsing the large file on the main thread.

**Do not use pandas.** Use only Python's standard library.

**Input**

- Parts: `C:/data/ir-weblogs/parts/weblog-1.log` through `weblog-20.log`
- Large file: `C:/data/ir-weblogs/large/weblog.log` (approximately 2–3 GB)

**Output**

- `C:/data/ir-weblogs/parsed_ir_weblogs.csv`

**Recommended time:** 90 minutes

## Requirements

1. Inspect the raw data, identify every field in a log record, and document the schema you discover.
2. Build one compiled regular expression with named groups for the complete log record. No regex pattern is supplied.
3. Write a function that parses one file line by line and returns valid records as tuples. Do not load a 2–3 GB file fully into memory.
4. Parse the 20 part files with `ThreadPoolExecutor`.
5. Set `max_workers` to the number of logical CPUs reported by the machine.
6. Submit one task per part file, keep the returned `Future` objects, and call `future.result()` to collect every result.
7. Combine all tuples and sort them by the actual event timestamp, not by the timestamp text.
8. Write the sorted records to CSV with a header using the `csv` module.
9. Parse the large `weblog.log` once on the main thread and measure its runtime. Do not write this duplicate result to CSV.
10. Compare the two timings and calculate the speedup.
11. Use the parsed tuples to answer the analytics questions at the end.

Keep malformed lines separately or count them. Never discard them silently.

## 1. Setup and validate the input

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
from datetime import datetime
from time import perf_counter
import csv
import os
import re

PARTS_DIR = Path('C:/data/ir-weblogs/parts')
LARGE_LOG = Path('C:/data/ir-weblogs/large/weblog.log')
CSV_PATH = Path('C:/data/ir-weblogs/parsed_ir_weblogs.csv')

# Build the list explicitly so that weblog-10.log does not come before weblog-2.log.
PART_FILES = [PARTS_DIR / f'weblog-{number}.log' for number in range(1, 21)]
MAX_WORKERS = os.cpu_count() or 1

missing = [path for path in [*PART_FILES, LARGE_LOG] if not path.is_file()]
print(f'Missing files: {len(missing)}')
print(f'Logical CPUs / max workers: {MAX_WORKERS}')

## 2. Discover and define the record format

A typical Apache combined-log row looks like this:

```text
83.149.9.216 - - [17/May/2015:10:05:03 +0000] "GET /index.html HTTP/1.1" 200 1024 "-" "Mozilla/5.0"
```

Inspect several rows from different part files. Identify every field, decide an appropriate field name and Python data type, and record your decisions in a short schema table in this notebook.

Then define `FIELD_NAMES`, your timestamp format, and one compiled regex with named capture groups. The tuple order must match `FIELD_NAMES`. Convert numeric values to numeric Python types and represent missing values consistently.

Your regex must be your own work. Test it against varied examples, including malformed or unusual rows.

In [ ]:
# TODO: inspect the logs and document the discovered schema in a Markdown cell.
# TODO: define FIELD_NAMES in the same order as the output tuple.
# TODO: define the timestamp format used by the data.
# TODO: compile a complete named-group regex as LOG_PATTERN.


## 3. Write one reusable file parser

Complete `parse_log_file`. Open the file inside the function, stream through it line by line, apply `LOG_PATTERN`, and return:

```text
(records, rejected_lines)
```

`records` is a list of tuples. Store each rejected line as `(line_number, line_text)`.

In [ ]:
def parse_log_file(path):
    records = []
    rejected_lines = []

    # TODO: stream the file, parse each line, convert data types,
    #       and append a tuple or rejected-line entry.
    pass

    return records, rejected_lines

Test the parser on the first part before starting the timed runs. The notebook intentionally provides no ready-made assertions: write your own checks based on the schema and the observed data.

In [ ]:
sample_records, sample_rejected = parse_log_file(PART_FILES[0])

# TODO: write and run your own data-quality and code-quality assertions.
print(sample_records[0] if sample_records else 'No valid record found')
print(f'Rejected in first part: {len(sample_rejected):,}')

### Assertion scenarios for your parser

Write assertions that cover at least these scenarios. Add checks that are appropriate for the data you discover.

- All 20 part files and the large file exist before processing begins.
- At least one valid record is parsed from a known non-empty input file.
- Every parsed record is a tuple and its length matches the discovered schema.
- Every tuple position has the expected Python type or the documented missing-value representation.
- A known valid sample line matches and preserves its field values.
- A malformed sample line is rejected and retains its original line number and text.
- The compiled regex contains the same named fields as the documented schema.
- The parser does not silently lose a line: valid plus rejected equals lines examined.

## 4. Parse the 20 files concurrently

Start the timer immediately before creating the executor and stop it only after all future results have been collected. Submit the files in numeric order. Keep a mapping from each future to its source path so that an exception can identify the failed file.

You must use `future.result()` in this section.

In [ ]:
parallel_start = perf_counter()

# TODO: create ThreadPoolExecutor(max_workers=MAX_WORKERS)
# TODO: submit parse_log_file once for each path in PART_FILES
# TODO: collect every future.result()
# TODO: combine records and rejected lines

parallel_seconds = perf_counter() - parallel_start

print(f'Parallel records: {len(parallel_records):,}')
print(f'Parallel rejected: {len(parallel_rejected):,}')
print(f'Parallel runtime: {parallel_seconds:.3f} seconds')

## 5. Sort and write the CSV

Sort `parallel_records` by the timestamp field discovered in your schema. Convert timestamp text to `datetime` in the sort key; do not rely on alphabetical sorting. Then write the header and all tuples with `csv.writer`.

Open the CSV with `newline=''` and an explicit encoding.

In [ ]:
# TODO: sort parallel_records by event timestamp
# TODO: create the output directory if required
# TODO: write FIELD_NAMES as the header, followed by all records
# TODO: write your own assertions for the output file

print(f'CSV written to {CSV_PATH}')

## 6. Parse the large file on the main thread

Call the same `parse_log_file` function directly—do not use an executor. Measure only the parsing call. Do not save these duplicate records to another CSV.

In [ ]:
single_start = perf_counter()
# TODO: call parse_log_file(LARGE_LOG) directly
single_seconds = perf_counter() - single_start

print(f'Single-thread records: {len(single_records):,}')
print(f'Single-thread rejected: {len(single_rejected):,}')
print(f'Single-thread runtime: {single_seconds:.3f} seconds')

## 7. Compare the results

Use your own assertions to confirm that both approaches processed the same number of valid and rejected lines. Calculate:

```text
speedup = single-thread runtime / parallel runtime
```

A value above `1.0` means the threaded run was faster. A value below `1.0` means it was slower. Report the measured result honestly; performance depends on the disk, operating system, Python runtime, regex work, and caching.

In [ ]:
# TODO: write assertions comparing the parallel and single-thread results

# TODO: calculate speedup
print(f'Speedup: {speedup:.2f}x')

# TODO: write two or three sentences interpreting your result.

### Assertion scenarios for the complete pipeline

Write assertions for these scenarios without hard-coding totals that you have not justified from the source data.

- Exactly 20 futures are created and each one produces a result.
- The configured worker count equals the machine's reported logical CPU count, with a safe fallback when it is unavailable.
- The combined parallel count equals the sum of the individual file results.
- Timestamps are in non-decreasing order after sorting.
- The CSV exists, its header matches the discovered schema, and its data-row count matches the valid-record count.
- Parallel and single-thread runs agree on valid and rejected counts.
- Important aggregate checks, such as the sum of method counts and status counts, equal the total valid-record count.
- Runtime values are positive and the speedup calculation uses the correct numerator and denominator.

## 8. Analytics without pandas

Use loops, comprehensions, `Counter`, dictionaries, sets, and tuple operations to answer:

1. How many valid events and rejected lines are there?
2. What are the counts for each HTTP method?
3. What are the counts for each status code?
4. Which five IP addresses made the most requests?
5. Which five URLs were requested most often?
6. How many responses are client errors (`400–499`) and server errors (`500–599`)?
7. What are the earliest and latest event timestamps?

Display each answer with a clear label.

In [ ]:
# TODO: complete the analytics using parallel_records only


## Submission checklist

- The notebook runs from top to bottom.
- The regex is compiled once and uses named groups.
- The parser streams files line by line and returns tuples.
- `ThreadPoolExecutor`, all logical CPUs, `Future` objects, and `future.result()` are used.
- Records are sorted by parsed event time.
- The CSV contains a header and all valid records.
- Both runtimes and the calculated speedup are shown.
- Analytics are completed without pandas.
- Rejected lines are reported, not silently lost.